In [1]:
!pip install openai pandas gradio matplotlib seaborn langchain langchain-openai python-docx fitz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.4/95.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.9/425.9 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.4/565.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.75
    Uninstalling langchain-core-0.3.75:
      Successfully uninstalled langchain-core-0.3.75


In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 77.0 MB/s eta 0:00:00


In [3]:
!pip install Streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.1 MB/s eta 0:00:00


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitz  # PyMuPDF
import docx
import json
from io import BytesIO
from openai import OpenAI
import base64
import re
import ast

# --- Model Wrapper ---
class ChatOpenAIWrapper:
    def __init__(self, api_key, base_url, model, max_tokens):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.max_tokens = max_tokens

    def ask(self, prompt):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=self.max_tokens
        )
        return response.choices[0].message.content.strip()

# --- Tool Functions ---
def plot_columns(df, columns):
    fig = sns.pairplot(df[columns])
    st.pyplot(fig)

def diagnose_dataset(df):
    issues = []
    if df.isnull().sum().sum() > 0:
        issues.append("⚠️ Missing values detected.")
    if df.duplicated().sum() > 0:
        issues.append("⚠️ Duplicate rows found.")
    if any(df.dtypes == 'object'):
        issues.append("⚠️ Non-numeric columns present.")
    return "\n".join(issues) if issues else "✅ No obvious problems detected."

def explain_dataset(df):
    shape = f"📐 Dataset has {df.shape[0]} rows and {df.shape[1]} columns."
    columns = f"🧩 Columns: {', '.join(df.columns)}"
    desc = df.describe(include='all').to_string()
    return f"{shape}\n{columns}\n\n📊 Summary:\n{desc}"

def extract_text(file):
    ext = file.name.split('.')[-1].lower()
    temp_path = "/tmp/" + file.name  # Save to temporary path

    with open(temp_path, "wb") as f:
        f.write(file.read())

    if ext == "txt":
        with open(temp_path, "r", encoding="utf-8") as f:
            return f.read()
    elif ext == "docx":
        doc = docx.Document(temp_path)
        return "\n".join([para.text for para in doc.paragraphs])
    elif ext == "pdf":
        doc = fitz.open(temp_path)
        return "\n".join([page.get_text() for page in doc])
    else:
        return ""

# --- Streamlit UI ---
def get_base64_image(image_path):
    with open(image_path, "rb") as image_file:
        encoded = base64.b64encode(image_file.read()).decode()
    return encoded




image_path = "/content/142+ Thousand Ai Bla.png"  # Update with your actual image path
base64_img = get_base64_image(image_path)

st.markdown(
    f"""
    <style>
    .stApp {{
        background-image: url("data:image/jpg;base64,{base64_img}");
        background-size: cover;
        background-repeat: no-repeat;
        background-attachment: fixed;
    }}
    </style>
    """,
    unsafe_allow_html=True
)


st.set_page_config(page_title="...SMART DATA ASSISTANT...", layout="centered")
st.markdown("""
<div style='text-align: center; padding: 20px; background: linear-gradient(to right, #8B4513, #A0522D); color: white; border-radius: 10px;'>
    <h1>🧠 DATA ANALYST ASSISTANT</h1>
    <p>Upload your dataset or document and describe what you want. Let Assistamt do the rest!</p>
</div>
""", unsafe_allow_html=True)



uploaded_file = st.file_uploader("📁 Upload CSV or Document", type=["csv", "txt", "docx", "pdf"])
instruction = st.text_area("📝 Describe the service you need", placeholder="What do you need to know from your file?")

# if st.button("⏎ ENTER ") and uploaded_file and instruction:
#     ext = uploaded_file.name.split('.')[-1].lower()
#     if ext == "csv":
#         df = pd.read_csv(uploaded_file)
#         prompt = f"""
#         You are a smart data assistant. The dataset has columns: {list(df.columns)}.
#         The user wants: "{instruction}".
#         Return a JSON object like: {{ "tool": "tool_name", "args": {{...}} }}.
#         Respond ONLY with a valid JSON object.
#         Do not include any explanations, markdown, or extra text.

#         """
#         model = ChatOpenAIWrapper(
#             api_key="6fc97f92d118491c83a909745a3bc001.i5InT5J1P2PfsYeg",
#             base_url="https://open.bigmodel.cn/api/paas/v4",
#             model="glm-4",
#             max_tokens=300
#         )
#         response = model.ask(prompt)
#         try:
#             parsed = json.loads(response)
#             tool = parsed["tool"]
#             args = parsed.get("args", {})
#             if tool == "plot_columns":
#                 plot_columns(df, args.get("columns", df.columns.tolist()[:2]))
#             elif tool == "diagnose_dataset":
#                 st.markdown(diagnose_dataset(df))
#             elif tool == "explain_dataset":
#                 st.text(explain_dataset(df))
#             else:
#                 st.error("Unknown tool requested.")
#         except Exception as e:
#             st.error(f"Error parsing model response: {e}")



if st.button("⏎ ENTER ") and uploaded_file and instruction:
    ext = uploaded_file.name.split('.')[-1].lower()

    # =========================
    # CASE 1: CSV FILE
    # =========================
    if ext == "csv":
      df = pd.read_csv(uploaded_file)

      # Convert the entire DataFrame to a list of dicts (all rows + columns)
      #  if the CSV is huge, this will make the prompt very large
      data_as_dict = df.to_dict(orient="records")

      prompt = f"""
      You are a smart data assistant. The dataset has {len(df)} rows and {len(df.columns)} columns.
      Columns: {list(df.columns)}
      Full data:
      {data_as_dict}
      Make sure you go through the given dataset and understand it very well.

      The user wants: "{instruction}".
      Think deeply before answering to make sure that all the answers you give are very true and suggest the important information or questions when necessary.
      You can answer in plain text, answer in possible short words and make the plain text beautiful and friendly,
      make the user feel welcomed and use simple english words to understand. If the user asks for a chart, describe what to plot and
      plot for the user if possible. if the user explains what kind of the graph he or she needs, plot it if possible and beautify it.
      """

      model = ChatOpenAIWrapper(
          api_key="6fc97f92d118491c83a909745a3bc001.i5InT5J1P2PfsYeg",
          base_url="https://open.bigmodel.cn/api/paas/v4",
          model="glm-4",
          max_tokens=2000  # increase to handle large outputs
      )

      try:
          response = model.ask(prompt)
      except Exception as e:
          st.error(f"Model call failed: {e}")
          st.stop()

      if not response or not str(response).strip():
          st.error("Empty response from model.")
          st.stop()

      st.markdown("### 🧠 RESPONSE ⬇️")
      st.write(response)

      # Optional: auto-plot if instruction contains 'plot' or 'chart'
      if "plot" in instruction.lower() or "chart" in instruction.lower():
          st.markdown("### Visualise")
          st.line_chart(df[df.columns[:2]])

    # =========================
    # CASE 2: NON-CSV FILE
    # =========================
    else:
        text = extract_text(uploaded_file)
        prompt = f"""The user uploaded a document, make sure you go through the document very well and understand it. Content:\n{text}\n\nInstruction: {instruction}.
                 beautify the plain text when answering and make it short and answer in simple and friendly language for the user to understand
                 and feel welcomed. Think deeply and make sure the answers you give are very true."""

        model = ChatOpenAIWrapper(
            api_key="6fc97f92d118491c83a909745a3bc001.i5InT5J1P2PfsYeg",
            base_url="https://open.bigmodel.cn/api/paas/v4",
            model="glm-4",
            max_tokens=500
        )

        try:
            response = model.ask(prompt)
        except Exception as e:
            st.error(f"Model call failed: {e}")
            st.stop()

        st.markdown("### 🧠 RESPONSE ⬇️")
        st.text(response if response is not None else "")

Writing app.py


In [8]:
!ngrok authtoken 30H0ukoxS2nlBYGl1FUTmCh7XLk_5rxNHmsyuosY4xC6GcW9V

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
!pip -q install pyngrok
from pyngrok import ngrok

def launch_website():
  try:
    if ngrok.get_tunnels():
      ngrok.kill()
    tunnel = ngrok.connect()

    print("Click this link to try your web app:")
    print(tunnel.public_url)

    !streamlit run --server.port 80 app.py >/dev/null # Connect to the URL through Port 80 (>/dev/null hides outputs)

  except KeyboardInterrupt:
    ngrok.kill()

print("Done!")

Done!


In [9]:
launch_website()

Click this link to try your web app:
https://59a7390a19bd.ngrok-free.app
